# Proyecto 2 — Análisis Exploratorio

**Curso:** CC3084 — Data Science · Universidad del Valle de Guatemala
**Semestre:** II — 2026

**Reto (Tema 3):** *A Step Ahead of Drought: Forecasting Global Water Storage Challenge*
organizado por la **ITU (International Telecommunication Union)** junto con agencias
del Sistema de Naciones Unidas, alojado en la plataforma Zindi.

**Categoría:** Series de tiempo con datos satelitales.

**Integrantes:**

1. ______________________________
2. ______________________________
3. ______________________________
4. ______________________________

---

## 1. Introducción

Este cuaderno desarrolla el análisis exploratorio de datos (EDA) del reto
*A Step Ahead of Drought*. El objetivo de esta etapa **no es construir el modelo
de pronóstico**, sino entender el fenómeno, la estructura del conjunto de datos y
los patrones que después permitirán plantear un modelo con criterio.

El documento está organizado de la siguiente manera. Las secciones 2 a 5 plantean
el problema: el contexto físico y satelital, la situación problemática que da
origen al reto, el problema científico y los objetivos. Las secciones 6 a 11
describen y preparan los datos. Las secciones 12 a 20 constituyen el análisis
exploratorio propiamente dicho, con énfasis en el comportamiento temporal por
tratarse de una serie de tiempo. Las secciones 21 a 23 resumen los hallazgos, las
conclusiones y los siguientes pasos.

### Nota sobre las fuentes

A lo largo del cuaderno se distingue de forma explícita entre dos tipos de
afirmación:

- **Evidencia del conjunto de datos.** Todo lo que se sostiene con una tabla, una
  estadística o una gráfica calculada en este mismo cuaderno.
- **Información externa.** Contexto tomado de la documentación oficial del reto o
  de literatura científica sobre GRACE, TWS y sequías. Aparece siempre con su
  referencia y se recopila en la sección de referencias.

Ninguna conclusión del análisis se apoya únicamente en información externa.

## 2. Contexto del problema

Esta sección corresponde a la investigación previa solicitada en la guía del
proyecto. Su información proviene de fuentes externas —documentación del reto y
literatura científica— y sirve para saber **qué buscar** durante el análisis
exploratorio.

### 2.1 ¿Qué fenómeno se está estudiando?

Se estudia la variación temporal del **agua almacenada en los continentes** y su
relación con la aparición de condiciones de sequía. No se trata únicamente de la
lluvia: se trata del balance acumulado entre el agua que entra a un territorio y
el agua que sale de él.

### 2.2 ¿Qué significa *Global Water Storage* en este reto?

El reto trabaja con el **Total Water Storage (TWS)**, definido en la propia
descripción de la competencia como *todo el agua almacenada sobre y bajo la
superficie terrestre*, lo que incluye:

| Componente         | Descripción                                          |
| ------------------ | ---------------------------------------------------- |
| Agua subterránea   | Acuíferos                                            |
| Humedad del suelo  | Agua retenida en la zona no saturada                 |
| Agua superficial   | Ríos, lagos, embalses, humedales                     |
| Nieve y hielo      | Acumulación estacional y glaciares                   |

El TWS se mide como una **anomalía**: no es un volumen absoluto, sino la
desviación respecto a una línea base histórica, expresada habitualmente en
centímetros o milímetros de **lámina de agua equivalente**. Un valor negativo
indica que hay menos agua almacenada que en el periodo de referencia.

Esta característica es importante para el análisis: una variable de anomalía está
**centrada alrededor de cero por construcción**, y esa es una expectativa que el
EDA debe verificar en los datos reales en lugar de darla por sentada.

### 2.3 El papel de los datos satelitales

El TWS a escala global se obtiene de las misiones **GRACE** (*Gravity Recovery and
Climate Experiment*, 2002–2017) y **GRACE-FO** (*GRACE Follow-On*, desde 2018).
Estas misiones no observan el agua directamente: miden variaciones muy pequeñas
del **campo gravitatorio terrestre**. Como el agua tiene masa, un cambio en la
cantidad de agua almacenada en una región cambia localmente la gravedad, y de esa
señal se deriva el TWS.

Esto tiene tres consecuencias prácticas que condicionan el análisis:

1. **Resolución espacial gruesa.** El producto es representativo de escalas del
   orden de cientos de kilómetros, no de una parcela o una cuenca pequeña.
2. **Resolución temporal mensual.** La serie es mensual, no diaria.
3. **Es una medida integrada en profundidad.** Suma agua superficial y subterránea,
   por lo que responde con más inercia que la lluvia o la humedad superficial del
   suelo.

### 2.4 ¿Por qué importa anticipar las sequías?

La sequía es un fenómeno de instalación lenta: cuando resulta visible en los
cultivos, en los caudales o en los pozos, el déficit lleva meses acumulándose.
Disponer de una estimación anticipada del almacenamiento de agua permite tomar
decisiones —restricciones de riego, gestión de embalses, alertas tempranas—
mientras todavía tienen efecto.

El TWS es particularmente útil aquí porque, al integrar el agua subterránea,
detecta **sequía hidrológica**, que es la que afecta el abastecimiento y que los
indicadores de respuesta rápida no capturan por sí solos.

### 2.5 ¿Qué se intenta pronosticar y por qué es un problema de series de tiempo?

De acuerdo con la descripción oficial del reto, la tarea consiste en predecir el
**TWS del mes siguiente** (`t+1`) a partir de información disponible en el mes `t`.
La motivación es la **latencia**: los productos derivados de GRACE se publican
con un retraso de aproximadamente 2 a 3 meses, de modo que el estado hídrico
"actual" no se conoce cuando se necesita.

Es un problema de series de tiempo por tres razones:

- La variable objetivo está **indexada en el tiempo** con frecuencia mensual y
  ordenada; el orden de las observaciones tiene significado.
- Existe **dependencia temporal**: el almacenamiento de un mes está fuertemente
  condicionado por el del mes anterior, dado que el agua se acumula y se agota de
  forma gradual.
- Se espera **estacionalidad** ligada al ciclo anual de lluvias, deshielo y
  evapotranspiración.

Formalmente es un problema de **regresión supervisada sobre datos
espacio-temporales**: para cada celda geográfica y cada mes se predice un valor
continuo. La métrica de evaluación del reto es el **RMSE**.

### 2.6 Covariables consideradas en el reto

La documentación del reto indica que, además del propio TWS, el conjunto de datos
incorpora indicadores climáticos con menor latencia que sirven como covariables:

- **SPEI** (*Standardised Precipitation-Evapotranspiration Index*) en varias
  escalas de acumulación (1, 3, 6 y 12 meses). Es un índice estandarizado que
  combina precipitación y evapotranspiración potencial. Los valores negativos
  indican déficit hídrico. Las escalas cortas reflejan sequía meteorológica y
  agrícola; las escalas largas se aproximan más a la sequía hidrológica.
- **Humedad del suelo** cercana a la superficie, que responde con rapidez a la
  lluvia y a la demanda evaporativa.

La expectativa razonable —que el análisis debe **verificar, no asumir**— es que las
escalas largas del SPEI se asocien más al TWS que las escalas cortas, precisamente
por la inercia del almacenamiento subterráneo.

> Las columnas concretas, sus nombres y sus unidades se verifican directamente
> sobre los archivos en la sección 8. En este cuaderno no se describe ninguna
> variable que no se haya observado en los datos.

## 3. Situación problemática

El agua dulce disponible para los ecosistemas y para la actividad humana no está
distribuida de manera uniforme ni es constante en el tiempo. Su disponibilidad
depende del balance entre lo que se recibe por precipitación y lo que se pierde por
evapotranspiración y escorrentía, y ese balance se acumula durante meses o años en
el suelo, en los cuerpos de agua superficiales y en los acuíferos. Cuando el
balance permanece negativo durante un periodo prolongado, el resultado es una
**sequía hidrológica**: una reducción sostenida del agua efectivamente almacenada
en un territorio.

El problema práctico es que esta reducción **no se observa con facilidad**. La
lluvia se mide bien, pero la lluvia por sí sola no describe el estado del sistema:
una región puede recibir precipitación normal y seguir agotándose si la demanda
evaporativa es alta o si la extracción de agua subterránea excede la recarga.
Medir directamente el agua almacenada requeriría una red de observación
—piezómetros, aforos, sensores de humedad— que a escala global no existe con
cobertura ni homogeneidad suficientes.

Aquí es donde los **datos satelitales** cambian el planteamiento. Las misiones
GRACE y GRACE-FO permiten estimar el almacenamiento total de agua de forma
indirecta, a partir de las variaciones del campo gravitatorio terrestre, con
cobertura global y de manera consistente en el tiempo. Por primera vez se dispone
de una variable comparable entre regiones que integra agua superficial y
subterránea, y que resulta adecuada para caracterizar el estado hídrico de grandes
territorios.

Sin embargo, esta solución trae consigo una limitación operativa. Según la
descripción del reto, los productos derivados de GRACE se publican con un
**retraso de aproximadamente 2 a 3 meses**. Ese desfase es determinante: para
monitoreo y alerta temprana se necesita conocer el estado actual del sistema, y lo
que está disponible es el estado de hace un trimestre. La información existe, pero
llega tarde para sostener una decisión.

De ahí surge la dificultad que motiva el reto. Anticipar el almacenamiento futuro
de agua no es trivial por varias razones que el análisis exploratorio debe
examinar:

- El comportamiento del agua almacenada **no es homogéneo en el espacio**. Una
  región tropical, una zona árida, una cuenca dominada por deshielo y una región
  con fuerte extracción de agua subterránea responden de maneras distintas, de
  modo que un patrón global promedio puede ocultar dinámicas opuestas.
- La señal combina al menos tres componentes —**tendencia de largo plazo,
  estacionalidad anual y variabilidad interanual**— que se superponen y que
  conviene separar antes de modelar.
- Las covariables disponibles con baja latencia, como el SPEI y la humedad del
  suelo, describen sobre todo la parte superficial y de respuesta rápida del ciclo
  hidrológico, mientras que el TWS incluye componentes de respuesta lenta. Su
  capacidad de anticipar el TWS es, por tanto, una **pregunta empírica**.
- Los registros satelitales presentan **interrupciones y valores faltantes**. En
  particular, entre el fin de la misión GRACE y el inicio de GRACE-FO existe un
  hueco en la serie, y su presencia debe verificarse en los datos del reto porque
  condiciona qué se puede modelar y con qué evidencia.

El análisis exploratorio busca precisamente caracterizar estos elementos con
evidencia, en lugar de darlos por conocidos.

## 4. Problema científico

> **¿Qué patrones temporales, estacionales y espaciales presenta la anomalía de
> almacenamiento total de agua (TWS) en el conjunto de datos del reto, y qué
> variables satelitales y climáticas disponibles en el mes actual muestran mayor
> asociación con el TWS del mes siguiente, de manera que puedan sustentar la
> construcción de un modelo de pronóstico a un mes de horizonte?**

La pregunta se puede responder con los datos disponibles porque cada uno de sus
componentes se traduce en un análisis concreto y verificable:

| Componente de la pregunta       | Cómo se responde                                                        |
| ------------------------------- | ----------------------------------------------------------------------- |
| Patrones temporales             | Serie agregada, medias móviles y comparación entre periodos (sección 16) |
| Patrones estacionales           | Perfil mensual, boxplots por mes y descomposición temporal (sección 17)  |
| Patrones espaciales             | Agregación por latitud y por región, y mapas de la rejilla (sección 18)  |
| Asociación entre variables      | Matriz de correlación y gráficos de dispersión (secciones 19 y 20)       |
| Utilidad para el pronóstico     | Asociación de la variable objetivo con el TWS actual y sus rezagos       |

Se trata de una pregunta **descriptiva y asociativa**, no causal. A lo largo del
cuaderno se habla de asociación o de correlación observada, nunca de causa: que
dos variables se muevan juntas no demuestra que una determine a la otra.

## 5. Objetivos

### 5.1 Objetivo general

Caracterizar, mediante un análisis exploratorio reproducible, el comportamiento
temporal, estacional y espacial de la anomalía de almacenamiento total de agua
(TWS) y de sus covariables en el conjunto de datos del reto
*A Step Ahead of Drought*, con el fin de identificar los patrones y las variables
que sustenten la construcción posterior de un modelo de pronóstico a un mes de
horizonte.

### 5.2 Objetivos específicos

1. **Describir** la estructura del conjunto de datos: número de observaciones y de
   variables, tipo de cada una, cobertura temporal y cobertura espacial,
   documentando la clasificación de las variables en identificadores, temporales,
   geográficas, satelitales y objetivo.

2. **Evaluar** la calidad de los datos cuantificando valores faltantes,
   observaciones duplicadas y valores extremos, y determinando si su presencia
   responde a patrones temporales o espaciales antes de tomar cualquier decisión
   de tratamiento.

3. **Analizar** el comportamiento temporal de la variable objetivo mediante
   estadística descriptiva, medias móviles y descomposición de la serie, para
   determinar la existencia de tendencia, estacionalidad y periodos anómalos.

4. **Comparar** el comportamiento del almacenamiento de agua entre distintas
   ubicaciones geográficas, para determinar si el patrón temporal global es
   representativo o si oculta dinámicas regionales diferentes.

5. **Determinar** el grado de asociación entre la variable objetivo, el
   almacenamiento del mes actual y las covariables climáticas disponibles,
   utilizando matrices de correlación y gráficos de dispersión.

6. **Identificar** las variables, transformaciones y estructuras temporales
   candidatas a ser utilizadas en la etapa posterior de modelado, con base en la
   evidencia obtenida en los objetivos anteriores.

Los seis objetivos son medibles: cada uno se cierra con una tabla, una gráfica o
un estadístico concreto dentro del cuaderno, y en la sección 22 se retoma cada uno
para verificar que fue cumplido.

## 6. Importación de librerías

Se utilizan las mismas librerías empleadas en los ejemplos y laboratorios del
curso: `pandas` y `numpy` para el manejo de datos, `matplotlib` y `seaborn` para
las visualizaciones, y `statsmodels` para la descomposición de la serie de tiempo
y la prueba de Dickey-Fuller, tal como se hizo en el ejemplo de series de tiempo
visto en clase. No se incorporan librerías adicionales que no aporten al análisis.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

# Opciones de despliegue: tablas legibles sin truncar columnas relevantes.
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Estilo de gráficos consistente en todo el cuaderno.
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 160
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Semilla para que cualquier muestreo sea reproducible.
SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

print(f"pandas      {pd.__version__}")
print(f"numpy       {np.__version__}")
print(f"matplotlib  {plt.matplotlib.__version__}")
print(f"seaborn     {sns.__version__}")

pandas      3.0.2
numpy       2.2.3
matplotlib  3.10.1
seaborn     0.13.2


In [2]:
# Rutas relativas al proyecto: el cuaderno se ejecuta desde la raíz del repositorio.
RAIZ = Path.cwd()
CRUDOS = RAIZ / "data" / "raw"
PROCESADOS = RAIZ / "data" / "processed"
FIGURAS = RAIZ / "figures"
TABLAS = RAIZ / "outputs" / "tablas"

for carpeta in (PROCESADOS, FIGURAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

print(f"Raíz del proyecto : {RAIZ}")
print(f"Datos crudos      : {CRUDOS.relative_to(RAIZ)}")
print(f"Figuras           : {FIGURAS.relative_to(RAIZ)}")


def guardar_figura(nombre):
    'Guarda la figura activa en figures/ para reutilizarla en informe y presentación.'
    plt.savefig(FIGURAS / f"{nombre}.png", bbox_inches="tight")

Raíz del proyecto : /Users/javiervalladares/Library/Mobile Documents/com~apple~CloudDocs/Octavo Semestre/Datos/Proyecto2
Datos crudos      : data/raw
Figuras           : figures


---

> **Estado del cuaderno.** Las secciones 1 a 6 están completas. Las secciones 7 en
> adelante (carga de datos, descripción, limpieza y análisis exploratorio) se
> construyen sobre los archivos reales del reto, que deben colocarse en
> `data/raw/` siguiendo las instrucciones de `data/raw/README.md`. No se incluye
> aquí ningún análisis con datos simulados ni resultados supuestos.